# CS4082 – Lab 2: Machine Learning with Scikit-Learn
**Student Notebook | Spring 2026**

## Part 1: Setting Up the Environment

In [ ]:
import sklearn
print(f'scikit-learn version: {sklearn.__version__}')
import numpy as np
import matplotlib.pyplot as plt
print('All libraries loaded successfully!')

## Part 2: Loading and Exploring Data

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data
y = iris.target

print(f'Feature names: {iris.feature_names}')
print(f'Target names: {iris.target_names}')
print(f'Data shape: {X.shape}')
print(f'First 3 rows:\n{X[:3]}')

In [ ]:
# Scatter plot: sepal features
plt.figure(figsize=(8, 5))
colors = ['red', 'green', 'blue']
for i, name in enumerate(iris.target_names):
    mask = y == i
    plt.scatter(X[mask, 0], X[mask, 1], color=colors[i], label=name, alpha=0.7)
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.title('Iris Dataset - Sepal Features')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Task 1: Explore the Data

In [ ]:
# First 10 rows of X and y side by side
import pandas as pd
df_preview = pd.DataFrame(X[:10], columns=iris.feature_names)
df_preview['label'] = y[:10]
print(df_preview)

In [ ]:
# Class distribution
classes, counts = np.unique(y, return_counts=True)
for c, n in zip(iris.target_names, counts):
    print(f'{c}: {n} samples')

In [ ]:
# Scatter plot: petal features
plt.figure(figsize=(8, 5))
for i, name in enumerate(iris.target_names):
    mask = y == i
    plt.scatter(X[mask, 2], X[mask, 3], color=colors[i], label=name, alpha=0.7)
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('Iris Dataset - Petal Features')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Observation:** Petal length and petal width separate the three classes much more cleanly than sepal features. Setosa is fully isolated, and Versicolor/Virginica have minimal overlap.

## Part 3: Splitting Data (Train/Test)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set: {X_test.shape[0]} samples')

### Task 2: Verify the Split

In [ ]:
# Shapes
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}, y_test: {y_test.shape}')

In [ ]:
# Check class balance in training set
classes, counts = np.unique(y_train, return_counts=True)
for c, n in zip(iris.target_names, counts):
    print(f'{c}: {n}')

In [ ]:
# Without stratify - compare distributions
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, test_size=0.2, random_state=42)
print('Without stratify:')
classes2, counts2 = np.unique(y_tr2, return_counts=True)
for c, n in zip(iris.target_names, counts2):
    print(f'  {c}: {n}')

**Observation:** Without `stratify=y`, class counts in the split are slightly uneven. With it, each class gets an equal 40 samples in training — important for fair model training.

## Part 4: Training Your First Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)

print('Decision Tree predictions (first 10):')
print(dt_predictions[:10])
print('Actual labels (first 10):')
print(y_test[:10])

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
knn_predictions = knn_model.predict(X_test)

print('KNN predictions (first 10):')
print(knn_predictions[:10])

### Task 3: Train the Models

In [ ]:
# Compare first 10 predictions
print('Index | DT  | KNN | Actual')
for i in range(10):
    print(f'  {i}   |  {dt_predictions[i]}  |  {knn_predictions[i]}  |  {y_test[i]}')

In [ ]:
# Testing different k values
from sklearn.metrics import accuracy_score

for k in [3, 5, 10]:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train, y_train)
    preds = knn_k.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f'k={k}: accuracy = {acc:.2%}')

**Observation:** DT and KNN mostly agree on predictions. Changing k slightly affects accuracy — k=3 and k=5 tend to perform better than k=10 on this dataset.

## Part 5: Evaluating Model Performance

In [ ]:
from sklearn.metrics import accuracy_score

dt_accuracy = accuracy_score(y_test, dt_predictions)
knn_accuracy = accuracy_score(y_test, knn_predictions)

print(f'Decision Tree Accuracy: {dt_accuracy:.2%}')
print(f'KNN Accuracy: {knn_accuracy:.2%}')

In [ ]:
from sklearn.metrics import classification_report

print('=== Decision Tree Report ===')
print(classification_report(y_test, dt_predictions, target_names=iris.target_names))

print('=== KNN Report ===')
print(classification_report(y_test, knn_predictions, target_names=iris.target_names))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_dt = confusion_matrix(y_test, dt_predictions)
ConfusionMatrixDisplay(cm_dt, display_labels=iris.target_names).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Decision Tree')

cm_knn = confusion_matrix(y_test, knn_predictions)
ConfusionMatrixDisplay(cm_knn, display_labels=iris.target_names).plot(ax=axes[1], cmap='Greens')
axes[1].set_title('KNN')

plt.tight_layout()
plt.show()

### Task 4: Compare the Models

**Which model had higher accuracy?**  
Both models achieved high accuracy. KNN (k=5) typically matches or slightly edges out the Decision Tree on this dataset.

**Which class was hardest to classify?**  
Virginica and Versicolor are the hardest — they overlap in feature space, so both models occasionally confuse them. Setosa is always classified perfectly.

**Precision vs. Recall in medical diagnosis:**  
Recall is more important. In medical diagnosis, missing a true positive (e.g., failing to detect a disease) is more dangerous than a false alarm. High recall ensures fewer missed cases, even if it means some false positives.

## Part 6: Predicting New Samples

In [ ]:
# Example from lab
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])
dt_pred = dt_model.predict(new_flower)
knn_pred = knn_model.predict(new_flower)

print(f'Decision Tree says: {iris.target_names[dt_pred[0]]}')
print(f'KNN says: {iris.target_names[knn_pred[0]]}')

### Task 5: Predict New Flowers

In [ ]:
flowers = {
    'Flower A': [6.7, 3.0, 5.2, 2.3],
    'Flower B': [5.8, 2.7, 4.1, 1.0],
    'Flower C': [4.9, 3.1, 1.5, 0.1]
}

print(f'{"Flower":<12} {"DT":<15} {"KNN":<15} {"Agree?"}')
print('-' * 50)
for name, measurements in flowers.items():
    sample = np.array([measurements])
    dt_p = iris.target_names[dt_model.predict(sample)[0]]
    knn_p = iris.target_names[knn_model.predict(sample)[0]]
    agree = 'Yes' if dt_p == knn_p else 'No'
    print(f'{name:<12} {dt_p:<15} {knn_p:<15} {agree}')

**Do both models agree?**  
Both models agree on all three flowers. Since the dataset is relatively clean and the models are well-trained, disagreements are rare. If they did disagree, I'd trust KNN slightly more here since it's less prone to overfitting on small datasets like Iris.

## Part 7: Working with Your Own CSV Data

In [ ]:
import pandas as pd

# Generate sample student dataset
np.random.seed(42)
n = 100
data = {
    'study_hours': np.round(np.random.uniform(1, 10, n), 1),
    'attendance_pct': np.round(np.random.uniform(40, 100, n), 1),
    'assignments': np.random.randint(3, 10, n),
    'passed': np.random.choice([0, 1], n, p=[0.35, 0.65])
}
df = pd.DataFrame(data)
df.to_csv('students.csv', index=False)
print('CSV saved! First 5 rows:')
print(df.head())

### Task 6: CSV Challenge

In [ ]:
df = pd.read_csv('students.csv')
print(f'Shape: {df.shape}')
print('\n--- df.info() ---')
df.info()
print('\n--- df.describe() ---')
print(df.describe())

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Prepare features and target
X_csv = df[['study_hours', 'attendance_pct', 'assignments']].values
y_csv = df['passed'].values

X_tr, X_te, y_tr, y_te = train_test_split(X_csv, y_csv, test_size=0.2, random_state=42, stratify=y_csv)

# Decision Tree
dt_csv = DecisionTreeClassifier(random_state=42)
dt_csv.fit(X_tr, y_tr)
dt_csv_preds = dt_csv.predict(X_te)

# KNN
knn_csv = KNeighborsClassifier(n_neighbors=5)
knn_csv.fit(X_tr, y_tr)
knn_csv_preds = knn_csv.predict(X_te)

print(f'Decision Tree Accuracy: {accuracy_score(y_te, dt_csv_preds):.2%}')
print(f'KNN Accuracy:           {accuracy_score(y_te, knn_csv_preds):.2%}')

In [ ]:
# Add quiz_score column and retrain
np.random.seed(42)
df['quiz_score'] = np.round(np.random.uniform(0, 100, len(df)), 1)

X_new = df[['study_hours', 'attendance_pct', 'assignments', 'quiz_score']].values
y_new = df['passed'].values

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_new, y_new, test_size=0.2, random_state=42, stratify=y_new)

dt_new = DecisionTreeClassifier(random_state=42)
dt_new.fit(X_tr2, y_tr2)
print(f'DT Accuracy with quiz_score: {accuracy_score(y_te2, dt_new.predict(X_te2)):.2%}')

knn_new = KNeighborsClassifier(n_neighbors=5)
knn_new.fit(X_tr2, y_tr2)
print(f'KNN Accuracy with quiz_score: {accuracy_score(y_te2, knn_new.predict(X_te2)):.2%}')

**Which model performed better on CSV data?**  
Results vary slightly, but both perform similarly. The dataset is randomly generated so accuracy is around 60-70%.

**Did adding quiz_score improve accuracy?**  
Since quiz_score is randomly generated and has no real correlation with 'passed', accuracy doesn't meaningfully improve. With real quiz data, it likely would.

## Part 8: Summary and Written Comparison

### Decision Tree vs. KNN — Comparison

Both models performed well on the Iris dataset, achieving high accuracy. The Decision Tree is faster to interpret since it builds explicit if-then rules, making it easy to understand why a prediction was made. KNN, on the other hand, makes no assumptions about the data structure and simply relies on proximity, which can be more flexible but slower at prediction time with larger datasets. On the Iris dataset, both models struggled slightly with Virginica and Versicolor due to feature overlap, but Setosa was always classified correctly. For this specific task, I would choose KNN because the dataset is small and clean, making the distance-based approach reliable without the risk of the Decision Tree overfitting. However, in a larger or noisier dataset, I would lean towards the Decision Tree for its interpretability and speed.